# Playground Series S6E4 - Baseline Models

This notebook starts the modeling phase for the irrigation-need competition. It uses the EDA findings to build transparent baseline models, evaluate them with stratified validation, and create a first submission file.

## 1. Setup

Load the Kaggle Python stack, define shared constants, and configure evaluation settings. The notebook is intentionally dependency-light and will skip optional models if their packages are unavailable.

In [ ]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler
from catboost import CatBoostClassifier

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', '{:,.5f}'.format)

sns.set_theme(style='whitegrid', context='notebook')
VIRIDIS_CMAP = 'viridis'
RANDOM_STATE = 42
N_SPLITS = 3
RUN_CROSS_VALIDATION = False
MAKE_SUBMISSION = True

## 2. Data Loading

Find the Kaggle competition files, load train/test data, and infer the ID and target columns from the sample submission.

In [ ]:
INPUT_ROOT = Path('/kaggle/input')

if not INPUT_ROOT.exists():
    raise RuntimeError('This notebook is intended to run on Kaggle, where /kaggle/input is available.')

def find_file(filename: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(f'Could not find {filename} under {INPUT_ROOT}. Attach the competition data.')
    if len(matches) > 1:
        print(f'Found multiple {filename} files. Using: {matches[0]}')
    return matches[0]

train_path = find_file('train.csv')
test_path = find_file('test.csv')
sample_submission_path = find_file('sample_submission.csv')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

target_col = sample_submission.columns[-1]
id_col = sample_submission.columns[0]

print(f'train shape: {train.shape}')
print(f'test shape: {test.shape}')
print(f'target column: {target_col}')
print(f'id column: {id_col}')
display(train[target_col].value_counts(normalize=True).mul(100).rename('target_pct').to_frame())

## 3. Feature Preparation

The first baseline uses the raw competition features. A second feature set adds a small number of EDA-driven interactions that were strongly linked to the rare `High` class.

### 3.1 Base Features

Separate predictors from ID and target columns, then identify numeric and categorical feature groups.

In [ ]:
base_feature_cols = [c for c in train.columns if c not in {id_col, target_col}]
numeric_cols = train[base_feature_cols].select_dtypes(include=np.number).columns.tolist()
categorical_cols = [c for c in base_feature_cols if c not in numeric_cols]

print(f'Base features: {len(base_feature_cols)}')
print(f'Numeric features: {len(numeric_cols)}')
print(f'Categorical features: {len(categorical_cols)}')
print('Categorical columns:', categorical_cols)

### 3.2 EDA Interaction Features

Add compact interaction features that reflect the strongest EDA findings: growth stage combined with mulching, water source, and irrigation type. These are simple string interactions so tree and linear baselines can both use them.

In [ ]:
INTERACTION_PAIRS = [
    ('Crop_Growth_Stage', 'Mulching_Used'),
    ('Crop_Growth_Stage', 'Water_Source'),
    ('Crop_Growth_Stage', 'Irrigation_Type'),
]

def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for left, right in INTERACTION_PAIRS:
        if left in out.columns and right in out.columns:
            out[f'{left}__x__{right}'] = out[left].astype(str) + '__' + out[right].astype(str)
    return out

train_fe = add_interaction_features(train)
test_fe = add_interaction_features(test)

feature_cols = [c for c in train_fe.columns if c not in {id_col, target_col}]
numeric_cols_fe = train_fe[feature_cols].select_dtypes(include=np.number).columns.tolist()
categorical_cols_fe = [c for c in feature_cols if c not in numeric_cols_fe]

print(f'Features with interactions: {len(feature_cols)}')
print('Added interactions:', [c for c in feature_cols if '__x__' in c])

## 4. Validation Design

Use stratified splits because `High` is rare. Metrics include accuracy for leaderboard intuition, macro F1 and balanced accuracy for class balance, and log loss when probability estimates are available.

In [ ]:
X = train_fe[feature_cols]
y = train_fe[target_col]
X_test = test_fe[feature_cols]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = label_encoder.classes_.tolist()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=RANDOM_STATE,
)

print('Classes:', class_names)
print('Train split:', X_train.shape)
print('Validation split:', X_valid.shape)

## 5. Model Definitions

CatBoost is the primary baseline because it performed best in the first Kaggle run and handles categorical features natively. Simpler scikit-learn models remain in the comparison as sanity checks and fallback references.

### 5.1 Preprocessing Pipelines

Define reusable preprocessors for one-hot models and ordinal tree models.

In [ ]:
try:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
except TypeError:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse=True)

onehot_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_cols_fe),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', onehot),
        ]), categorical_cols_fe),
    ],
    remainder='drop',
)

ordinal_preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_cols_fe),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ]), categorical_cols_fe),
    ],
    remainder='drop',
)

### 5.2 Baseline Model Set

Create the model set for comparison. CatBoost is named as the primary candidate rather than an optional add-on.

In [ ]:
models = {
    'dummy_most_frequent': DummyClassifier(strategy='most_frequent'),
    'logistic_onehot': Pipeline([
        ('preprocess', onehot_preprocessor),
        ('model', LogisticRegression(max_iter=500, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)),
    ]),
    'random_forest_onehot': Pipeline([
        ('preprocess', onehot_preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=250,
            max_depth=18,
            min_samples_leaf=20,
            class_weight='balanced_subsample',
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]),
    'hist_gradient_boosting': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', HistGradientBoostingClassifier(
            learning_rate=0.08,
            max_iter=250,
            max_leaf_nodes=31,
            l2_regularization=0.05,
            random_state=RANDOM_STATE,
        )),
    ]),
    'catboost': CatBoostClassifier(
        iterations=600,
        learning_rate=0.06,
        depth=7,
        loss_function='MultiClass',
        eval_metric='TotalF1',
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
    ),
}

list(models)

## 6. Holdout Evaluation

Fit each baseline on the training split and evaluate on the stratified validation split. This provides quick feedback before running cross-validation.

### 6.1 Holdout Metrics

Fit each candidate model on the stratified training split and compare validation metrics.

In [ ]:
def fit_model(model, X_train, y_train, X_valid=None, y_valid=None):
    if model.__class__.__name__ == 'CatBoostClassifier':
        cat_indices = [X_train.columns.get_loc(c) for c in categorical_cols_fe]
        model.fit(X_train, y_train, cat_features=cat_indices, eval_set=(X_valid, y_valid) if X_valid is not None else None)
    else:
        model.fit(X_train, y_train)
    return model

def predict_proba_or_none(model, X_data):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X_data)
    return None

def evaluate_predictions(y_true, y_pred, y_proba=None):
    row = {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted'),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
    }
    if y_proba is not None:
        row['log_loss'] = log_loss(y_true, y_proba, labels=np.arange(len(class_names)))
    return row

holdout_rows = []
fitted_models = {}

for name, model in models.items():
    print(f'Fitting {name}...')
    fitted = fit_model(clone(model), X_train, y_train, X_valid, y_valid)
    y_pred = fitted.predict(X_valid)
    if y_pred.ndim > 1:
        y_pred = y_pred.reshape(-1)
    y_pred = y_pred.astype(int)
    y_proba = predict_proba_or_none(fitted, X_valid)
    row = evaluate_predictions(y_valid, y_pred, y_proba)
    row['model'] = name
    holdout_rows.append(row)
    fitted_models[name] = fitted

holdout_results = pd.DataFrame(holdout_rows).set_index('model').sort_values('macro_f1', ascending=False)
display(holdout_results)

### 6.2 Baseline Output Insights

The first executed baseline run showed CatBoost as the strongest holdout model: `0.9851` accuracy, `0.9711` macro F1, `0.9638` balanced accuracy, and `0.0602` log loss. Histogram gradient boosting was very close on macro F1 and slightly better on log loss, while random forest had the best balanced accuracy but worse log loss.

The key result is that CatBoost achieved strong minority-class performance without manual tuning: `High` precision was `0.9653`, recall was `0.9205`, and F1 was `0.9424`. This is strong enough to justify moving from baseline comparison into CatBoost-focused tuning and validation.

## 7. Model Diagnostics

Inspect the best holdout model in more detail. The confusion matrix and class report are especially important for the rare `High` class.

In [ ]:
best_model_name = holdout_results.index[0]
best_model = fitted_models[best_model_name]

valid_pred = best_model.predict(X_valid)
if valid_pred.ndim > 1:
    valid_pred = valid_pred.reshape(-1)
valid_pred = valid_pred.astype(int)

print('Best holdout model:', best_model_name)
print(classification_report(y_valid, valid_pred, target_names=class_names, digits=4))

cm = confusion_matrix(y_valid, valid_pred, labels=np.arange(len(class_names)))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap=VIRIDIS_CMAP, xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap=VIRIDIS_CMAP, xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('Row-Normalized Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 8. Model Interpretation

Inspect the best model to confirm whether it relies on the same drivers found in EDA. For CatBoost, feature importance is computed directly from the fitted model.

### 8.1 CatBoost Feature Importance

Rank the features used by the fitted CatBoost model and compare them with the EDA signal ranking.

In [ ]:
if best_model_name == 'catboost':
    importance = best_model.get_feature_importance(prettified=True)
    importance = importance.rename(columns={'Feature Id': 'feature', 'Importances': 'importance'})
else:
    importance = pd.DataFrame({'feature': [], 'importance': []})
    print(f'Feature importance is currently configured for CatBoost. Best model was: {best_model_name}')

if not importance.empty:
    display(importance.head(30))
    plt.figure(figsize=(10, 7))
    sns.barplot(
        data=importance.head(25),
        x='importance',
        y='feature',
        hue='feature',
        palette=VIRIDIS_CMAP,
        legend=False,
    )
    plt.title('CatBoost Feature Importance')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

### 8.2 Interpretation Notes

After running the notebook, compare the feature-importance ranking with the EDA findings. Agreement between model importance and EDA signal increases confidence that the model is learning stable irrigation patterns rather than artifacts.

## 9. Cross-Validation

Run stratified cross-validation for a more stable estimate. This can be disabled by setting `RUN_CROSS_VALIDATION = False` in the setup cell during quick iteration.

In [ ]:
cv_results = []

if RUN_CROSS_VALIDATION:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for name, model in models.items():
        for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y_encoded), start=1):
            print(f'CV {name} fold {fold}/{N_SPLITS}')
            X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
            y_tr, y_va = y_encoded[train_idx], y_encoded[valid_idx]
            fitted = fit_model(clone(model), X_tr, y_tr, X_va, y_va)
            pred = fitted.predict(X_va)
            if pred.ndim > 1:
                pred = pred.reshape(-1)
            pred = pred.astype(int)
            proba = predict_proba_or_none(fitted, X_va)
            row = evaluate_predictions(y_va, pred, proba)
            row.update({'model': name, 'fold': fold})
            cv_results.append(row)

    cv_results_df = pd.DataFrame(cv_results)
    display(cv_results_df)
    cv_summary = cv_results_df.groupby('model').agg(['mean', 'std']).sort_values(('macro_f1', 'mean'), ascending=False)
    display(cv_summary)
else:
    print('Cross-validation disabled.')

## 10. Final Training and Submission

Fit the best selected model on all training rows and write `submission.csv` for Kaggle. By default the selected model is the best holdout model; after cross-validation, update `selected_model_name` if another model is more reliable.

In [ ]:
selected_model_name = best_model_name
print('Selected model:', selected_model_name)

if MAKE_SUBMISSION:
    final_model = fit_model(clone(models[selected_model_name]), X, y_encoded)
    test_pred = final_model.predict(X_test)
    if test_pred.ndim > 1:
        test_pred = test_pred.reshape(-1)
    test_pred = test_pred.astype(int)
    test_labels = label_encoder.inverse_transform(test_pred)

    submission = sample_submission.copy()
    submission[target_col] = test_labels
    submission.to_csv('submission.csv', index=False)

    display(submission.head())
    display(submission[target_col].value_counts(normalize=True).mul(100).rename('prediction_pct').to_frame())
    print('Wrote submission.csv')
else:
    print('Submission creation disabled.')

## 11. Baseline Summary and Next Experiments

Use this section after running on Kaggle to record the best holdout/CV model, leaderboard score, minority-class behavior, and the next experiment ideas.

Current baseline interpretation from the latest run:

- CatBoost is the strongest baseline: `0.9851` accuracy, `0.9711` macro F1, `0.9638` balanced accuracy, and `0.0602` log loss.
- `High` class performance is strong for a first baseline: `0.9653` precision, `0.9205` recall, and `0.9424` F1.
- Histogram gradient boosting is close and has slightly better log loss, so it remains useful as a calibration sanity check.
- The prediction mix from the first CatBoost submission is close to train distribution: `59.23%` `Low`, `37.59%` `Medium`, and `3.18%` `High`.

Recommended next notebook: CatBoost tuning with stratified cross-validation, interaction-feature comparison, and focused tracking of macro F1, log loss, and `High` recall/precision.